<div style="background:linear-gradient(120deg,#EDE7FF 0%,#E3F0FF 55%,#E6FBF7 100%); color:#2A1B5C; padding:22px 26px; border-radius:14px; border:1px solid #DCD3FF; margin-bottom:8px;">

<h1 style="margin:0 0 6px 0; color:#3B1F7A">01 · Collecting public pharma data</h1>

<p style="margin:0 0 10px 0; font-size:1.1em; color:#4C3A8C">Eight public US sources, fetched with code: APIs, pagination, rate limits, retries and streaming</p>

| | |
|---|---|
| **Goal** | Gather every public source the platform needs, reproducibly, with one function call each |
| **Code** | `src/glp1_intel/data_collection/` (one module per source; this notebook calls them) |
| **Output** | `data/01_raw/<source>/` (raw, never edited) + `data/02_interim/nppes_glp1_prescribers.parquet` |
| **Skills** | REST APIs · JSON/XML · pagination · rate limits · retries · streaming · data provenance |

</div>

<div style="margin:6px 0 4px 0"><b>How to read this notebook:</b><br><span style="background:#F3EEFF; color:#3B1F7A; border-left:4px solid #8B5CF6; padding:3px 10px; border-radius:6px; margin-right:6px; display:inline-block; margin-bottom:4px">🟣 Section</span><span style="background:#FFF4E8; color:#7C2D12; border-left:4px solid #FB923C; padding:3px 10px; border-radius:6px; margin-right:6px; display:inline-block; margin-bottom:4px">💡 Concept</span><span style="background:#E9FBF8; color:#134E4A; border-left:4px solid #14B8A6; padding:3px 10px; border-radius:6px; margin-right:6px; display:inline-block; margin-bottom:4px">🛠️ What the next code does</span><span style="background:#EAF4FF; color:#1E3A8A; border-left:4px solid #3B82F6; padding:3px 10px; border-radius:6px; margin-right:6px; display:inline-block; margin-bottom:4px">📊 Reading the output</span><span style="background:#FFF0F3; color:#881337; border-left:4px solid #F43F5E; padding:3px 10px; border-radius:6px; margin-right:6px; display:inline-block; margin-bottom:4px">⚠️ Watch out</span><span style="background:#EAFBF1; color:#065F46; border-left:4px solid #10B981; padding:3px 10px; border-radius:6px; margin-right:6px; display:inline-block; margin-bottom:4px">🎯 Key takeaways</span></div>

<div style="border-left:6px solid #FB923C; background:#FFF4E8; color:#7C2D12; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

<b style="font-size:1.02em; letter-spacing:.3px">💡 Concept · What is an API, and why fetch data with code?</b>

An **API** (Application Programming Interface) lets a program ask a server for data instead of a person clicking "Download".
Most public data APIs are **REST** APIs: you send an HTTP `GET` request to a URL (the **endpoint**), add **query parameters**, and get back **JSON** (or XML).

```
GET https://api.fda.gov/drug/event.json ? search=patient.drug.openfda.generic_name:"tirzepatide" & limit=1
└──────────── endpoint ───────────────┘   └──────────────── query parameters ─────────────────────┘
```

**Why code instead of manual downloads?** It's **reproducible** (anyone can re-run it), **documented** (the query *is* the documentation), **refreshable** (next month: run again) and **auditable** (we log what we fetched and when).

</div>

<div style="border-left:6px solid #8B5CF6; background:#F3EEFF; color:#3B1F7A; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

## 0 · The eight sources at a glance

</div>

<div style="border-left:6px solid #FB923C; background:#FFF4E8; color:#7C2D12; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

<b style="font-size:1.02em; letter-spacing:.3px">💡 Concept · Source map</b>

| # | Source | Gives us | How we fetch it |
|---|---|---|---|
| 1 | **openFDA FAERS** | Side-effect report counts per drug | REST API, server-side `count=` |
| 2 | **openFDA drug labels** | Official label text (indications, warnings, adverse reactions) | REST API |
| 3 | **PubMed** | Research abstracts + authors | NCBI E-utilities, *history server* paging |
| 4 | **ClinicalTrials.gov** | Trials, sponsors, sites, investigators | REST API v2, *page-token* paging |
| 5 | **Medicaid SDUD** | Prescriptions per state per quarter (incl. **2026**) | DKAN datastore API, *offset* paging |
| 6 | **Open Payments** | Lilly / Novo payments to each doctor | **Streaming** the official bulk CSV |
| 7 | **NPPES registry** | Doctor credential, sex, practice ZIP, taxonomy | Bulk zip, streamed and filtered |
| 8 | **ADE Corpus v2** | Labelled adverse-drug-event sentences (for NLP) | Hugging Face `datasets` |

*(Part D 2024 was downloaded by hand and filtered in notebook 02.)*

</div>

<div style="border-left:6px solid #8B5CF6; background:#F3EEFF; color:#3B1F7A; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

## 1 · Five engineering ideas used by every fetcher

</div>

<div style="border-left:6px solid #FB923C; background:#FFF4E8; color:#7C2D12; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

<b style="font-size:1.02em; letter-spacing:.3px">💡 Concept · Pagination, rate limits, retries, idempotency, provenance</b>

**1 · Pagination: servers send big results in pages.** We met four different styles:

| Style | How it works | Where |
|---|---|---|
| **Offset** | `limit=500&offset=1000` → rows 1,000–1,499 | Medicaid, Open Payments API |
| **Page token** | Server returns `nextPageToken`; send it back for the next page | ClinicalTrials.gov |
| **History server** | Search once → server stores the result list → fetch it in batches | PubMed |
| **Server-side aggregation** | Ask for **counts** instead of raw records | openFDA `count=` |

**2 · Rate limits: be polite.** Free APIs cap requests (openFDA: 240/min; NCBI: 3/s). We `sleep()` between calls.

**3 · Retries with exponential backoff.** Networks fail. On a temporary error (429 "too many requests", 5xx server error) we wait 1 s, 2 s, 4 s… and try again.
→ `src/glp1_intel/data_collection/_http.py`

**4 · Idempotent + atomic writes.** Each fetcher **skips work that already exists** (safe to re-run), and writes to a `.tmp` file first, renaming it only when complete. A crash can never leave a half-file that looks finished.

**5 · Provenance.** Next to each download we save a small `*_meta.json`: the query, total matches and date. A reviewer may ask: *"How do you know where this number came from?"* → the meta file.

</div>

<div style="border-left:6px solid #14B8A6; background:#E9FBF8; color:#134E4A; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

<b style="font-size:1.02em; letter-spacing:.3px">🛠️ What the next code does</b>

1. Import the fetchers (one module per source) and the `show()` helper for light-blue tables.
2. Load `.env` (optional API keys; everything works without them, just more slowly).
3. Define `folder_size()`, a small helper to report how much disk each source uses.

</div>

In [1]:
# ── Imports ─────────────────────────────────────────────────────────────────
import json
from pathlib import Path

import duckdb
import pandas as pd
from dotenv import load_dotenv

from glp1_intel.paths import RAW, INTERIM
from glp1_intel.notebook_style import show
from glp1_intel.data_collection import (
    openfda,          # 1 + 2  FAERS counts, drug labels
    pubmed,           # 3      research abstracts
    clinicaltrials,   # 4      trials
    medicaid,         # 5      state drug utilisation
    open_payments,    # 6      industry payments
    nppes,            # 7      doctor registry
    ade_corpus,       # 8      NLP training data
)

load_dotenv()                                        # optional: OPENFDA_API_KEY, NCBI_API_KEY in .env


def folder_size(path: Path) -> str:
    """Total size of all files in a folder, as a readable string."""
    mb = sum(f.stat().st_size for f in path.rglob("*") if f.is_file()) / 1e6
    return f"{mb:,.1f} MB"

<div style="border-left:6px solid #8B5CF6; background:#F3EEFF; color:#3B1F7A; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

## 2 · openFDA FAERS: side-effect reports

</div>

<div style="border-left:6px solid #FB923C; background:#FFF4E8; color:#7C2D12; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

<b style="font-size:1.02em; letter-spacing:.3px">💡 Concept</b>

**FAERS** (FDA Adverse Event Reporting System) collects reports of suspected side effects from doctors, pharmacists, patients and manufacturers.
There are millions of individual reports, but we don't need them all: openFDA can **count on the server** (`count=patient.reaction.reactionmeddrapt.exact`) and send back just the totals.

Reaction names use **MedDRA**, the standard medical dictionary for adverse events (e.g. "NAUSEA", "VOMITING").

</div>

In [2]:
faers = openfda.fetch_faers_counts()                 # cached: re-downloads only if the file is missing
meta  = json.loads((RAW / "fda_faers_side_effect_reports/faers_meta.json").read_text())

totals = pd.DataFrame(meta["total_reports"].items(), columns=["drug", "total_reports"])
show(totals.sort_values("total_reports", ascending=False), f"Total FAERS reports per drug (fetched {meta['fetched_at']})")

drug,total_reports
tirzepatide,"160,663"
dulaglutide,"88,160"
semaglutide,"73,001"
exenatide,"52,314"
liraglutide,"50,704"
orforglipron,728


In [3]:
# ── Top-8 reported reactions per drug, side by side ─────────────────────────
reactions = faers[faers["count_type"] == "reactions"]
top = (reactions.sort_values("count", ascending=False)
                .groupby("drug").head(8)
                .assign(rank=lambda d: d.groupby("drug")["count"].rank(ascending=False, method="first").astype(int)))
top_table = top.pivot(index="rank", columns="drug", values="term").reset_index()
show(top_table[["rank", "tirzepatide", "semaglutide", "dulaglutide", "orforglipron"]], "Most-reported reactions (rank 1 = most frequent)")

rank,tirzepatide,semaglutide,dulaglutide,orforglipron
1,INCORRECT DOSE ADMINISTERED,NAUSEA,NAUSEA,NAUSEA
2,NAUSEA,OFF LABEL USE,INJECTION SITE PAIN,DRUG INEFFECTIVE
3,INJECTION SITE PAIN,VOMITING,BLOOD GLUCOSE INCREASED,CONSTIPATION
4,DIARRHOEA,DIARRHOEA,DIARRHOEA,VOMITING
5,EXTRA DOSE ADMINISTERED,DECREASED APPETITE,VOMITING,FATIGUE
6,OFF LABEL USE,CONSTIPATION,INCORRECT DOSE ADMINISTERED,WEIGHT INCREASED
7,VOMITING,WEIGHT DECREASED,INAPPROPRIATE SCHEDULE OF PRODUCT ADMINISTRATION,WEIGHT LOSS POOR
8,DRUG INEFFECTIVE,IMPAIRED GASTRIC EMPTYING,DRUG INEFFECTIVE,DIARRHOEA


In [4]:
# ── Serious vs non-serious (openFDA code: 1 = serious, 2 = not serious) ─────
serious = (faers[faers["count_type"] == "serious"]
           .pivot(index="drug", columns="term", values="count")
           .rename(columns={"1": "serious", "2": "not_serious"}))
serious["pct_serious"] = 100 * serious["serious"] / serious.sum(axis=1)
show(serious.reset_index().sort_values("pct_serious"), "Share of reports flagged serious (%)")

drug,serious,not_serious,pct_serious
orforglipron,20,708,2.7
tirzepatide,"28,502","132,118",17.7
dulaglutide,"22,898","65,252",26.0
exenatide,"14,629","37,685",28.0
liraglutide,"24,709","25,964",48.8
semaglutide,"36,179","36,792",49.6


In [5]:
# ── Reports received per quarter (the "trend" view) ─────────────────────────
per_q = (faers[faers["count_type"] == "reports_per_quarter"]
         .pivot(index="term", columns="drug", values="count")
         .reset_index().rename(columns={"term": "quarter"}))
show(per_q.tail(8), "FAERS reports received per quarter (last 8 quarters)")

quarter,dulaglutide,exenatide,liraglutide,orforglipron,semaglutide,tirzepatide
2024Q3,"1,838",36,401,0,"5,857","8,963"
2024Q4,"1,451",54,409,0,"2,480","9,489"
2025Q1,"1,531",37,783,0,"2,849","11,639"
2025Q2,"1,341",16,350,0,"2,695","15,612"
2025Q3,"1,918",14,545,1,"11,817","17,781"
2025Q4,"1,082",8,525,0,"2,652","17,270"
2026Q1,"1,129",4,822,0,"2,625","19,467"
2026Q2,"1,340",4,471,727,"9,378","19,881"


<div style="border-left:6px solid #3B82F6; background:#EAF4FF; color:#1E3A8A; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

<b style="font-size:1.02em; letter-spacing:.3px">📊 Reading the output</b>

- **Nausea** is in the top 3 for every GLP-1, as expected from the labels.
- For tirzepatide, **"Incorrect dose administered"** ranks #1: a signal about **pen and dosing errors** (and compounded versions), not about the molecule itself. That's a great "voice of the doctor" theme for reps.
- **Orforglipron (Foundayo)** already has **728 reports**, almost all received in **Q2 2026** right after approval: nausea, "drug ineffective", constipation. Only **2.7%** are flagged serious (tirzepatide: 18%).
- Tirzepatide reports keep rising every quarter (**9k in Q3 2024 → 20k in Q2 2026**), in line with fast-growing use.
- Semaglutide jumps in some quarters (e.g. 2025 Q3: 11.8k vs ~2.7k usual): batches of reports arriving together (for example from litigation or manufacturer back-filing), a reminder that FAERS measures **reporting**.

</div>

<div style="border-left:6px solid #F43F5E; background:#FFF0F3; color:#881337; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

<b style="font-size:1.02em; letter-spacing:.3px">⚠️ Watch out</b>

- FAERS reports are **voluntary and unverified**: counts reflect **reporting**, not true frequency. **No causality** and no rates without knowing how many patients use the drug.
- Fast-growing drugs and media attention produce more reports (the *stimulated reporting* effect).
- Semaglutide shows fewer reports than expected because openFDA's generic-name mapping misses many free-text product names. Later we'll add brand-name searches for a fairer comparison.

</div>

<div style="border-left:6px solid #8B5CF6; background:#F3EEFF; color:#3B1F7A; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

## 3 · openFDA drug labels: the official 'source of truth'

</div>

<div style="border-left:6px solid #FB923C; background:#FFF4E8; color:#7C2D12; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

<b style="font-size:1.02em; letter-spacing:.3px">💡 Concept</b>

A drug **label** (Structured Product Label, SPL) is the FDA-approved document: indications, dosing, warnings, adverse reactions, clinical studies.
Pharma reps may only make claims **consistent with the label** (no "off-label" promotion), so the label becomes the **knowledge base** our AI assistant must stay within (RAG guardrails, notebook 14).

</div>

In [6]:
counts = openfda.fetch_labels()                      # {brand: number of label versions found}

rows = []
for brand in counts:
    labels = json.loads((RAW / f"fda_drug_labels/label_{brand.lower()}.json").read_text(encoding="utf-8"))
    for lab in labels:
        rows.append({
            "brand":          brand,
            "manufacturer":   ", ".join(lab["openfda"].get("manufacturer_name", ["?"])),
            "effective_date": lab.get("effective_time"),
            "has_boxed_warning": "boxed_warning" in lab,
            "adverse_reactions_chars": len(" ".join(lab.get("adverse_reactions", []))),
        })
labels_df = pd.DataFrame(rows)
show(labels_df.sort_values(["brand", "effective_date"], ascending=[True, False]), "Label versions per brand")

brand,manufacturer,effective_date,has_boxed_warning,adverse_reactions_chars
Foundayo,Eli Lilly and Company,20260729,True,"9,843"
Mounjaro,Eli Lilly and Company,20260827,True,"14,922"
Mounjaro,A-S Medication Solutions,20260729,True,"14,072"
Mounjaro,A-S Medication Solutions,20260729,True,"14,072"
Mounjaro,A-S Medication Solutions,20260729,True,"14,072"
Mounjaro,A-S Medication Solutions,20260729,True,"14,072"
Mounjaro,A-S Medication Solutions,20260729,True,"14,072"
Mounjaro,A-S Medication Solutions,20260729,True,"14,072"
Ozempic,A-S Medication Solutions,20260730,True,"18,228"
Ozempic,"Novo Nordisk Pharmaceutical Industries, LP",20260130,True,"10,381"


<div style="border-left:6px solid #3B82F6; background:#EAF4FF; color:#1E3A8A; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

<b style="font-size:1.02em; letter-spacing:.3px">📊 Reading the output</b>

- Every brand has a label. **Foundayo** (approved April 2026) already has its own.
- Several brands appear more than once: **repackagers** (e.g. "A-S Medication Solutions") republish the label under their name.
- **Mounjaro** has Lilly's own label from Aug 2026 plus six repackager copies.
- `has_boxed_warning`: the injectable GLP-1s carry the FDA's strongest warning (thyroid C-cell tumours).
- `adverse_reactions_chars` shows how much text we'll later chunk for retrieval.

</div>

<div style="border-left:6px solid #F43F5E; background:#FFF0F3; color:#881337; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

<b style="font-size:1.02em; letter-spacing:.3px">⚠️ Watch out</b>

- For the knowledge base we'll keep **one label per brand**: the **manufacturer's own, most recent** version. Otherwise duplicate repackager copies would bias retrieval.
- **A trap we found:** the Novo Nordisk label returned for "Ozempic" is actually the **oral Rybelsus** label (route = ORAL, tagged with both brand names), and the newest Wegovy label here is a 2024 repackager copy. Never trust a search result's name alone: **check the route and manufacturer fields**. We'll fetch Novo's own Ozempic and Wegovy labels from DailyMed before building the RAG knowledge base.

</div>

<div style="border-left:6px solid #8B5CF6; background:#F3EEFF; color:#3B1F7A; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

## 4 · PubMed: research abstracts and authors

</div>

<div style="border-left:6px solid #FB923C; background:#FFF4E8; color:#7C2D12; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

<b style="font-size:1.02em; letter-spacing:.3px">💡 Concept</b>

NCBI **E-utilities** works in two steps:
1. `esearch` runs the query and (with `usehistory=y`) **stores the result list on NCBI's server**, returning a `WebEnv` key.
2. `efetch` downloads the stored list **200 records at a time** using that key.

We keep title, abstract, journal, year, MeSH terms (the controlled medical vocabulary) and **authors with affiliations**. Affiliations let us later find which US doctors publish on GLP-1s, a "key opinion leader" (KOL) signal.

</div>

In [7]:
pub_counts = pubmed.fetch_all()                      # cached per query

rows = []
for name in pub_counts:
    meta = json.loads((RAW / f"pubmed_research_papers/pubmed_{name}_meta.json").read_text())
    rows.append({"query": name, "pubmed_matches": meta["total_matches"], "downloaded": meta["downloaded"],
                 "search_term": meta["term"]})
show(pd.DataFrame(rows), "PubMed queries")

query,pubmed_matches,downloaded,search_term
orforglipron,146,144,orforglipron
tirzepatide,"2,606","2,594",tirzepatide AND 2022:2026[pdat]
semaglutide_obesity_diabetes,"3,789","3,780","semaglutide AND (obesity OR ""weight loss"" OR diabetes) AND 2024:2026[pdat]"
glp1_medicare_access,"2,401","2,382","(""GLP-1 receptor agonists"" OR GLP-1) AND (Medicare OR coverage OR access OR adherence OR prescribing) AND 2023:2026[pdat]"


In [8]:
# ── Papers per year, using DuckDB directly on the JSONL files ───────────────
per_year = duckdb.sql(f"""
    SELECT  query, year, COUNT(*) AS papers
    FROM    read_json_auto('{(RAW / "pubmed_research_papers").as_posix()}/pubmed_*.jsonl')
    WHERE   year BETWEEN '2022' AND '2026'
    GROUP BY ALL
""").df().pivot(index="year", columns="query", values="papers").reset_index()
show(per_year, "Papers per publication year")

year,glp1_medicare_access,orforglipron,semaglutide_obesity_diabetes,tirzepatide
2022,—,—,—,93
2023,187,13,—,195
2024,344,16,816,330
2025,776,30,"1,264",792
2026,"1,075",85,"1,700","1,183"


<div style="border-left:6px solid #3B82F6; background:#EAF4FF; color:#1E3A8A; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

<b style="font-size:1.02em; letter-spacing:.3px">📊 Reading the output</b>

- `downloaded` is slightly below `pubmed_matches` because a few records are not standard journal articles (e.g. book chapters) and are skipped by our parser.
- Research volume explodes: tirzepatide papers went from **93 (2022) to 1,183 (2026 so far)**, orforglipron from **13 to 85**. That's more than anyone can read, which is exactly the problem NLP solves.

</div>

<div style="border-left:6px solid #F43F5E; background:#FFF0F3; color:#881337; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

<b style="font-size:1.02em; letter-spacing:.3px">⚠️ Watch out</b>

Linking authors to doctors (NPI) is **fuzzy**: PubMed has no NPI, and names repeat ("J. Smith"). In notebook 03 we'll match on **last name + first initial + state** from the affiliation and report the match quality honestly.

</div>

<div style="border-left:6px solid #8B5CF6; background:#F3EEFF; color:#3B1F7A; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

## 5 · ClinicalTrials.gov: who runs GLP-1 trials, and where

</div>

<div style="border-left:6px solid #FB923C; background:#FFF4E8; color:#7C2D12; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

<b style="font-size:1.02em; letter-spacing:.3px">💡 Concept</b>

The v2 API returns each study as nested JSON "modules" (identification, status, design, sponsors, **locations**, **officials**).
Paging uses a **token**: each response includes `nextPageToken` until the last page. Site locations and investigator names are an engagement signal: doctors running trials tend to be early adopters.

</div>

In [9]:
ct_counts = clinicaltrials.fetch_all()


def flatten(study: dict) -> dict:
    """Pick the few fields we need from one nested study record."""
    ps = study["protocolSection"]
    return {
        "nct_id":   ps["identificationModule"]["nctId"],
        "status":   ps["statusModule"]["overallStatus"],
        "phase":    ", ".join(ps.get("designModule", {}).get("phases", []) or ["N/A"]),
        "sponsor":  ps["sponsorCollaboratorsModule"]["leadSponsor"]["name"],
        "us_sites": sum(loc.get("country") == "United States"
                        for loc in ps.get("contactsLocationsModule", {}).get("locations", [])),
    }


studies = pd.concat(
    pd.DataFrame([flatten(json.loads(line)) for line in open(RAW / f"clinicaltrials_gov_studies/ctgov_{drug}.jsonl", encoding="utf-8")])
      .assign(drug=drug)
    for drug in ct_counts
)
summary = (studies.groupby("drug")
                  .agg(studies=("nct_id", "count"),
                       recruiting=("status", lambda s: (s == "RECRUITING").sum()),
                       phase3=("phase", lambda s: s.str.contains("PHASE3").sum()),
                       us_sites=("us_sites", "sum"))
                  .reset_index().sort_values("studies", ascending=False))
show(summary, "Clinical trials per GLP-1 molecule")

drug,studies,recruiting,phase3,us_sites
semaglutide,785,169,184,"6,232"
liraglutide,514,25,112,"2,969"
tirzepatide,300,99,57,"2,488"
dulaglutide,142,9,36,"1,140"
orforglipron,53,9,27,951
cagrilintide,44,11,22,"1,030"
retatrutide,33,2,14,823


In [10]:
top_sponsors = (studies.groupby(["drug", "sponsor"]).size().rename("studies").reset_index()
                       .sort_values("studies", ascending=False).groupby("drug").head(2))
show(top_sponsors[top_sponsors["drug"].isin(["orforglipron", "tirzepatide", "semaglutide", "retatrutide"])],
     "Top sponsors per molecule")

drug,sponsor,studies
semaglutide,Novo Nordisk A/S,262
tirzepatide,Eli Lilly and Company,80
orforglipron,Eli Lilly and Company,49
retatrutide,Eli Lilly and Company,32
semaglutide,Brigham and Women's Hospital,19
tirzepatide,Brigham and Women's Hospital,11
orforglipron,Massachusetts General Hospital,2
retatrutide,Hudson Biotech,1


<div style="border-left:6px solid #3B82F6; background:#EAF4FF; color:#1E3A8A; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

<b style="font-size:1.02em; letter-spacing:.3px">📊 Reading the output</b>

- **Semaglutide** has the most studies (785; 169 still recruiting); **tirzepatide** has 300 (99 recruiting); **orforglipron (53, of which 27 phase 3)** and **retatrutide (33)** are the new pipeline.
- `us_sites` counts US trial locations: thousands of sites, each a hospital or practice we can later link to doctors by city and state.
- Lilly leads 49 of 53 orforglipron studies and all retatrutide studies; Novo Nordisk leads 262 semaglutide studies; academic centres (Brigham and Women's, Mass General) run investigator-initiated trials.

</div>

<div style="border-left:6px solid #8B5CF6; background:#F3EEFF; color:#3B1F7A; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

## 6 · Medicaid SDUD: the most recent prescription trends (2024 → 2026)

</div>

<div style="border-left:6px solid #FB923C; background:#FFF4E8; color:#7C2D12; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

<b style="font-size:1.02em; letter-spacing:.3px">💡 Concept</b>

Part D stops at 2024, but **Medicaid State Drug Utilization Data** is published **quarterly** and already covers **Q1 2026**.
It's not doctor-level, only state × quarter × package (NDC), but it's our best public **"what is happening now"** signal, for example how fast Zepbound grows after coverage changes.

</div>

In [11]:
sdud_counts = medicaid.fetch_all()
sdud = pd.concat(pd.read_csv(RAW / f"medicaid_state_drug_utilization/sdud_{y}_glp1.csv", dtype=str) for y in medicaid.YEARS)
print("Raw rows per year:", sdud_counts)

# ── Cleaning, step by step ──────────────────────────────────────────────────
sdud = sdud[sdud["state"] != "XX"]                                   # 1. drop national-total rows (avoid double count)
sdud["number_of_prescriptions"] = pd.to_numeric(sdud["number_of_prescriptions"])   # 2. text → number (suppressed → NaN)
sdud["brand"] = sdud["product_name"].str.upper().map(                # 3. "MOUNJARO 1", "MOUNJARO 7" → "MOUNJARO"
    lambda name: next((b for b in medicaid.BRANDS if name.startswith(b)), name))
sdud["period"] = sdud["year"] + "Q" + sdud["quarter"]

trend = (sdud.pivot_table(index="period", columns="brand", values="number_of_prescriptions", aggfunc="sum")
             .reset_index())
show(trend[["period", "OZEMPIC", "MOUNJARO", "TRULICITY", "ZEPBOUND", "WEGOVY", "RYBELSUS"]],
     "Medicaid prescriptions per quarter (all states, unsuppressed rows)", precision=0)

Raw rows per year: {2024: 10364, 2025: 12473, 2026: 3365}


period,OZEMPIC,MOUNJARO,TRULICITY,ZEPBOUND,WEGOVY,RYBELSUS
2024Q1,"681,652","174,973","555,961","22,415","121,248","60,627"
2024Q2,"777,477","189,994","489,754","43,384","207,454","60,509"
2024Q3,"921,773","242,179","540,329","77,764","376,857","66,314"
2024Q4,"896,713","257,879","513,781","133,123","397,258","56,747"
2025Q1,"897,369","275,114","498,310","241,065","441,885","52,308"
2025Q2,"910,315","315,502","493,790","340,696","471,038","47,459"
2025Q3,"980,988","385,938","487,276","531,415","593,864","52,845"
2025Q4,"906,373","438,473","460,138","608,430","487,096","47,567"
2026Q1,"845,813","481,351","383,648","365,721","236,826","45,354"


<div style="border-left:6px solid #3B82F6; background:#EAF4FF; color:#1E3A8A; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

<b style="font-size:1.02em; letter-spacing:.3px">📊 Reading the output</b>

Each row is a quarter; each column is the total Medicaid prescriptions for a brand.
- **Zepbound grew 27×**, from 22k (2024 Q1) to 608k (2025 Q4) prescriptions per quarter, and **Mounjaro nearly tripled** (175k → 481k). Ozempic plateaued around 900k and Trulicity is declining.
- **Q1 2026 drops sharply** for Zepbound (−40%) and Wegovy (−51%). Two possible reasons: states **ending obesity coverage** for budget reasons, or **late reporting** (the newest quarter is often incomplete). We'll test both in notebook 04 before drawing any conclusion.
- Coverage decides volume: a preview of what the **July 2026 Medicare Bridge** could do for obesity prescriptions.

</div>

<div style="border-left:6px solid #F43F5E; background:#FFF0F3; color:#881337; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

<b style="font-size:1.02em; letter-spacing:.3px">⚠️ Watch out</b>

- **State `XX`** rows are national totals: always drop them before summing states.
- **Suppression:** counts under 11 are hidden (`suppression_used = true`), so the totals above are slight **under**-counts.
- Product names are **truncated to 10 characters** ("LIRAGLUTID"), which is why we match on the brand prefix.
- `FFSU` = fee-for-service, `MCOU` = managed care. We keep both; they're different payment channels, not duplicates.

</div>

<div style="border-left:6px solid #8B5CF6; background:#F3EEFF; color:#3B1F7A; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

## 7 · Open Payments: industry payments to doctors

</div>

<div style="border-left:6px solid #FB923C; background:#FFF4E8; color:#7C2D12; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

<b style="font-size:1.02em; letter-spacing:.3px">💡 Concept</b>

Under the US **Sunshine Act**, drug makers must publicly report payments to doctors: meals, speaker fees, consulting, travel, research.
For us it's a proxy for **engagement**: which doctors a company already works with.

**Why streaming?** Each year's official CSV is ~8 GB (15M+ payments). The web API returns only 500 rows per request and slows down on deep pages (10–40 s each, so hours for 2M rows).
So we read the official CSV **while it downloads**, keep only Lilly / Novo Nordisk rows, and never store the full file. That's the same idea as DuckDB in notebook 02: **filter while reading**.

This takes ~1.5–2 hours per year, so it runs from the command line, not inside the notebook:
```bash
python -m glp1_intel.data_collection.open_payments
```

</div>

In [12]:
op_files = sorted((RAW / "open_payments_industry_to_doctor").glob("open_payments_general_*_lilly_novo.parquet"))
if not op_files:
    print("Not downloaded yet: run `python -m glp1_intel.data_collection.open_payments` in a terminal.")
else:
    op = " UNION ALL ".join(f"SELECT * FROM read_parquet('{f.as_posix()}')" for f in op_files)
    by_company = duckdb.sql(f"""
        SELECT  Program_Year                                                     AS year,
                CASE WHEN Applicable_Manufacturer_or_Applicable_GPO_Making_Payment_Name ILIKE '%lilly%'
                     THEN 'Eli Lilly' ELSE 'Novo Nordisk' END                    AS company,
                COUNT(*)                                                         AS payments,
                COUNT(DISTINCT Covered_Recipient_NPI)                            AS doctors,
                SUM(TRY_CAST(Total_Amount_of_Payment_USDollars AS DOUBLE)) / 1e6 AS total_usd_m
        FROM    ({op})
        GROUP BY ALL ORDER BY year, company
    """).df()
    display(show(by_company, "Payments by company and year"))

    by_nature = duckdb.sql(f"""
        SELECT  Nature_of_Payment_or_Transfer_of_Value                           AS nature,
                COUNT(*)                                                         AS payments,
                ROUND(AVG(TRY_CAST(Total_Amount_of_Payment_USDollars AS DOUBLE)), 0) AS avg_usd
        FROM    ({op})
        GROUP BY ALL ORDER BY payments DESC LIMIT 8
    """).df()
    show(by_nature, "What kinds of payments? (top 8)")

year,company,payments,doctors,total_usd_m
2024,Eli Lilly,"554,538","122,671",54.7
2024,Novo Nordisk,"496,384","110,650",28.4
2025,Eli Lilly,"432,923","122,513",55.7
2025,Novo Nordisk,"513,765","125,868",98.9


<div style="border-left:6px solid #3B82F6; background:#EAF4FF; color:#1E3A8A; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

<b style="font-size:1.02em; letter-spacing:.3px">📊 Reading the output</b>

- Most payments by count are small **food and beverage** items (a ~US$20 lunch during a rep visit), so the **number of payments** is a rough proxy for **rep contact**.
- Large average amounts come from **consulting** and **speaker fees**: these doctors are likely **key opinion leaders**.

</div>

<div style="border-left:6px solid #F43F5E; background:#FFF0F3; color:#881337; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

<b style="font-size:1.02em; letter-spacing:.3px">⚠️ Watch out</b>

Open Payments ≠ sales effort. It shows **payments**, not visits; many rep visits involve no payment at all. And payments ≠ influence: this is an **engagement** proxy only, and we describe it neutrally.

</div>

<div style="border-left:6px solid #8B5CF6; background:#F3EEFF; color:#3B1F7A; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

## 8 · NPPES: who is each prescriber?

</div>

<div style="border-left:6px solid #FB923C; background:#FFF4E8; color:#7C2D12; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

<b style="font-size:1.02em; letter-spacing:.3px">💡 Concept</b>

**NPPES** is the national registry behind every NPI. The API answers one doctor per request (~5 per second, so ~11 hours for our 200k doctors).
The monthly **bulk file** (1.2 GB zip, ~11 GB unzipped) is faster, but our disk is nearly full, so we **stream the CSV from inside the zip** and keep only our 199,831 GLP-1 prescribers. The zip is never unpacked.

**Taxonomy codes** (e.g. `207RE0101X` = Endocrinology) are the official specialty classification, more precise than Part D's free-text specialty.

</div>

In [13]:
npi = nppes.build()                                  # cached in data/02_interim/
part_d_npis = duckdb.sql(f"SELECT COUNT(DISTINCT Prscrbr_NPI) FROM read_parquet('{(INTERIM / 'part_d_2024_glp1.parquet').as_posix()}')").fetchone()[0]

match = pd.DataFrame([{
    "part_d_prescribers": part_d_npis,
    "found_in_nppes":     npi["npi"].nunique(),
    "match_pct":          100 * npi["npi"].nunique() / part_d_npis,
    "deactivated":        npi["deactivation_date"].notna().sum(),
}])
show(match, "NPPES match rate for our GLP-1 prescribers")

part_d_prescribers,found_in_nppes,match_pct,deactivated
"199,831","199,830",100.0,"1,291"


In [14]:
profile = pd.concat([
    npi["sex"].value_counts(dropna=False).rename_axis("value").reset_index().assign(field="sex"),
    npi["credential"].str.upper().str.replace(".", "", regex=False).value_counts().head(6)
       .rename_axis("value").reset_index().assign(field="credential (top 6)"),
    npi["taxonomy_1"].value_counts().head(6).rename_axis("value").reset_index().assign(field="taxonomy (top 6)"),
])[["field", "value", "count"]]
show(profile, "Who are the prescribers?")

field,value,count
sex,F,"115,440"
sex,M,"83,714"
sex,—,668
sex,X,6
sex,U,2
credential (top 6),MD,"98,290"
credential (top 6),DO,"20,277"
credential (top 6),PA-C,"9,784"
credential (top 6),NP,"7,636"
credential (top 6),FNP,"7,116"


<div style="border-left:6px solid #3B82F6; background:#EAF4FF; color:#1E3A8A; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

<b style="font-size:1.02em; letter-spacing:.3px">📊 Reading the output</b>

- **199,830 of 199,831 matched (100%)**. **1,291** have since been **deactivated** (retired, deceased or merged): important for targeting, since nobody should plan visits to them.
- **58% are women**, because nurse practitioners and physician assistants (many FNP, APRN, PA-C credentials) are a big share of GLP-1 prescribers.
- Top taxonomy codes: `207Q00000X` = **Family Medicine**, `207R00000X` = **Internal Medicine**, the same primary-care dominance we saw in notebook 02.
- `practice_zip5` (not shown) gives each doctor a 5-digit ZIP code: our key for **maps** and **territory alignment** later.

</div>

<div style="border-left:6px solid #8B5CF6; background:#F3EEFF; color:#3B1F7A; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

## 9 · ADE Corpus v2: training data for side-effect detection

</div>

<div style="border-left:6px solid #FB923C; background:#FFF4E8; color:#7C2D12; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

<b style="font-size:1.02em; letter-spacing:.3px">💡 Concept</b>

To detect side effects in rep visit notes (notebook 12), a model needs **labelled examples**.
ADE Corpus v2 has sentences from medical case reports, labelled **1 = mentions an adverse drug event** or **0 = doesn't**, plus a version marking *which words* are the drug and the effect (for **named entity recognition**).

</div>

In [15]:
ade_sizes = ade_corpus.fetch()
cls = pd.read_parquet(RAW / "ade_corpus_training_data/ade_corpus_v2_classification.parquet")
rel = pd.read_parquet(RAW / "ade_corpus_training_data/ade_corpus_v2_drug_ade_relation.parquet")

display(show(pd.DataFrame(ade_sizes.items(), columns=["config", "rows"]), "ADE Corpus v2 configurations"))
display(show(cls["label"].value_counts().rename_axis("label").reset_index()
        .assign(meaning=lambda d: d["label"].map({1: "adverse event", 0: "no adverse event"})), "Class balance"))
show(rel[["text", "drug", "effect"]].head(4), "Examples with drug and effect marked")

config,rows
Ade_corpus_v2_classification,"23,516"
Ade_corpus_v2_drug_ade_relation,"6,821"
Ade_corpus_v2_drug_dosage_relation,279


label,count,meaning
0,"16,695",no adverse event
1,"6,821",adverse event


text,drug,effect
Intravenous azithromycin-induced ototoxicity.,azithromycin,ototoxicity
"Immobilization, while Paget's bone disease was present, and perhaps enhanced activation of dihydrotachysterol by rifampicin, could have led to increased calcium-release into the circulation.",dihydrotachysterol,increased calcium-release
Unaccountable severe hypercalcemia in a patient treated for hypoparathyroidism with dihydrotachysterol.,dihydrotachysterol,hypercalcemia
METHODS: We report two cases of pseudoporphyria caused by naproxen and oxaprozin.,naproxen,pseudoporphyria


<div style="border-left:6px solid #3B82F6; background:#EAF4FF; color:#1E3A8A; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

<b style="font-size:1.02em; letter-spacing:.3px">📊 Reading the output</b>

- **~29% positive (ADE) vs 71% negative**: an **imbalanced** dataset. Accuracy alone would mislead (always guessing "no ADE" scores 71%), so we'll judge models on **precision, recall and F1**.
- The relation set tells us exactly which words are the drug ("azithromycin") and the effect ("ototoxicity"), the basis for **token classification** with BIO tags.

</div>

<div style="border-left:6px solid #8B5CF6; background:#F3EEFF; color:#3B1F7A; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

## 10 · Inventory & summary

</div>

In [16]:
inventory = pd.DataFrame([
    {"folder": p.name, "files": sum(1 for f in p.iterdir() if f.is_file() and f.name != ".gitkeep"), "size": folder_size(p)}
    for p in sorted(RAW.iterdir()) if p.is_dir()
])
show(inventory, "data/01_raw: what we have (all git-ignored)")

folder,files,size
ade_corpus_training_data,3,2.1 MB
clinicaltrials_gov_studies,7,74.5 MB
fda_drug_labels,9,7.5 MB
fda_faers_side_effect_reports,2,0.1 MB
medicaid_state_drug_utilization,3,2.3 MB
medicare_part_d_prescribers,1,"4,426.4 MB"
nppes_doctor_registry,1,"1,159.7 MB"
open_payments_industry_to_doctor,5,49.6 MB
pubmed_research_papers,8,30.4 MB


<div style="border-left:6px solid #F43F5E; background:#FFF0F3; color:#881337; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

<b style="font-size:1.02em; letter-spacing:.3px">⚠️ Limitations to carry forward</b>

1. **Different time windows:** Part D = 2024, Medicaid = up to Q1 2026, Open Payments = 2024–2025, FAERS = to mid-2026. Always state the period next to a number.
2. **Different coverage:** Medicare (65+) vs Medicaid (low income) vs everyone (FAERS). None covers commercial insurance at doctor level.
3. **Linking is imperfect:** NPI joins (Part D ↔ NPPES ↔ Open Payments) are exact; PubMed and trial investigators need fuzzy name matching.
4. **Snapshots change:** APIs return today's data; the `*_meta.json` files record what we fetched and when.

</div>

<div style="border-left:6px solid #10B981; background:#EAFBF1; color:#065F46; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

<b style="font-size:1.02em; letter-spacing:.3px">🎯 Key takeaways</b>

**Done:** 8 public sources fetched by reusable, resumable code; small validated summaries for each.
**Next:** Notebook 03 joins Part D + NPPES + Open Payments (+ trial and publication signals) on **NPI** into the one-row-per-doctor **HCP 360** table.

**Check your understanding**
- *How did you handle an API that returns only 500 rows per request?* Pagination, and when that was too slow, streaming the bulk file with filtering while reading.
- *What is exponential backoff?* Retrying after 1, 2, 4… seconds so temporary failures don't kill the job and the server isn't hammered.
- *What makes a pipeline idempotent?* Re-running gives the same result and skips finished work; atomic `.tmp` → rename writes prevent half-files.
- *Why is FAERS not a measure of side-effect frequency?* Voluntary, unverified reporting with no denominator; it reflects reporting behaviour.
- *Why keep only one label per brand?* Repackager duplicates would bias retrieval in RAG.
- *Why is ADE detection judged on F1, not accuracy?* The classes are imbalanced (29% positive).

</div>